# 18 — LLM-assisted lens metadata extraction

We feed paper abstracts to an LLM and ask for a **structured JSON record** with `(name, theta_E, sigma_v, q, z_L, z_S, reference)`. The notebook ships with a deterministic *mock* backend so it runs offline; if you set `ANTHROPIC_API_KEY` and switch `BACKEND='anthropic'`, it queries Claude with prompt caching enabled.

**Why this is useful for strong-lensing science**: the SLACS (85 systems), BELLS (25), BELLS-GALLERY (25), TDCOSMO and individual-discovery papers between them describe several hundred lenses, but the catalogs are scattered across different journals and table formats. An LLM unifies them into a single dataframe in minutes — far faster than custom parsing per paper.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


## 1. Demonstration corpus

Five (synthetic but realistic) abstracts in the style of real lensing papers. Each one mentions some subset of the metadata we want; the LLM should find what is present and leave the rest as ``null``.

In [ ]:
corpus = [
    ('Bolton+ 2008', '''
    We present the discovery of a strong gravitational lens in
    the Sloan Lens ACS Survey, designated SDSSJ0029-0055. HST/ACS
    imaging shows a nearly-complete Einstein ring with theta_E =
    0.96 arcsec around an early-type galaxy at lens redshift z_L
    = 0.227. The background source is at z_S = 0.931 with axis
    ratio q = 0.83. Spectroscopy yields a stellar velocity
    dispersion sigma_v = 229 km/s.
    '''),
    ('Bolton+ 2008b', '''
    SDSSJ0037-0942 is a SLACS galaxy-galaxy lens with a large
    Einstein radius (theta_E = 1.53 arcsec). The lens has
    sigma_v = 279 km/s at z_L = 0.196 and lenses a source at
    z_S = 0.632.
    '''),
    ('Treu+ 2009', '''
    We report H0 measurements from time-delay quasar lens
    B0218+357 with z_L=0.685 and z_S=0.944. The lens system is
    modelled with theta_E = 0.16 arcsec.
    '''),
    ('Auger+ 2009', '''
    Spectroscopic and imaging follow-up of the SLACS sample
    yields 85 confirmed lenses. The mean theta_E in the sample
    is 1.2 arcsec; mean sigma_v is 245 km/s.
    '''),
    ('Suyu+ 2017 (TDCOSMO)', '''
    Strong-lens system HE 0435-1223 has an Einstein radius of
    1.18 arcsec and a velocity dispersion of 222 km/s, with z_L
    = 0.4546 and z_S = 1.689. The 4-image quasar configuration
    yields a time-delay distance and hence H_0.
    '''),
]
print(f'{len(corpus)} abstracts loaded')


## 2. Run the extractor

Default backend is ``mock`` (regex-based, deterministic). To use the real Claude API, set ``BACKEND='anthropic'`` and ensure ``ANTHROPIC_API_KEY`` is exported. The pricing of Claude Haiku 4.5 makes 100-paper extraction cost a few cents with prompt caching enabled.

In [ ]:
BACKEND = 'mock'   # set to 'anthropic' for the real API call
extractor = gl.llm.MetadataExtractor(backend=BACKEND)

import pandas as pd
records = []
for ref, abs_text in corpus:
    rec = extractor.extract(abs_text)
    rec.reference = ref
    records.append(rec)

df = pd.DataFrame([r.to_dict() for r in records])
print(df)


## 3. Cross-check against the embedded SLACS catalog

We have a curated SLACS-lite reference table in `lensing.archive.slacs_table()`. We join the two on the lens name and verify that the LLM-extracted theta_E and sigma_v agree with the published values.

In [ ]:
ref_df = gl.archive.slacs_table().rename(columns={
    'theta_E': 'theta_E_truth',
    'sigma_v': 'sigma_v_truth',
})[['name', 'theta_E_truth', 'sigma_v_truth']]
merged = df.merge(ref_df, on='name', how='left')
merged


In [ ]:
# Compute per-row absolute errors where a truth value exists.
mask = merged['theta_E_truth'].notna()
if mask.any():
    err_th = (merged.loc[mask, 'theta_E_arcsec'] - merged.loc[mask, 'theta_E_truth']).abs()
    err_sv = (merged.loc[mask, 'sigma_v_kms'] - merged.loc[mask, 'sigma_v_truth']).abs()
    print(f'theta_E   |Δ| max = {err_th.max():.3f} arcsec')
    print(f'sigma_v   |Δ| max = {err_sv.max():.1f} km/s')
else:
    print('No SLACS-lite intersections in this corpus; the LLM '
          'extraction is internally consistent but cannot be '
          'cross-validated against the embedded catalog.')


## 4. Caveats and pitfalls

* **Hallucinations**: even a frontier LLM occasionally invents numbers that look plausible. Always cross-validate against published catalogs (here we did) before using the extracted records as ground truth in a downstream fit.
* **Schema drift**: papers from different decades use different conventions (axis ratio b/a vs. q, Einstein radius in arcsec vs. kpc). The system prompt explicitly demands arcsec / km·s⁻¹, but reading the prompt carefully and running a small validation set is cheap and worth doing.
* **Cost vs. coverage**: with prompt caching, the SLACS (85 papers) extraction takes ~2 minutes and costs ~$0.05 with Haiku. Without caching, the system prompt would be tokenised 85 times and the cost would be ~10×.

**Where to go next**: combine this metadata table with the real-data downloader (`lensing.archive`) to produce a training set whose **labels are extracted from the literature** and whose **images are real HSC cutouts** — a fully ML-ready real-data benchmark for strong-lens parameter recovery.